In [ ]:
!nvidia-smi

Thu Apr 16 06:33:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U "transformers>=5.5.0" accelerate bitsandbytes
!pip install -q librosa soundfile pillow
!pip install -q -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 15.2 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login

# Option A: interactive — paste token when prompted
login()

# Option B: Colab secrets (uncomment if using)
# from google.colab import userdata
# login(token=userdata.get('HF_TOKEN'))

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/gemma-4-E2B-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,     # T4 prefers fp16 over bf16
    device_map="auto",
).eval()

print(f"Model loaded. Device: {model.device}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded. Device: cuda:0
VRAM used: 10.21 GB


In [ ]:
def generate(messages, max_new_tokens=512, thinking=False):
    """Run the model on a chat-formatted messages list."""
    # Optionally prepend the thinking control token to the system prompt
    if thinking:
        sys_msg = {"role": "system", "content": [{"type": "text", "text": "<|think|>"}]}
        messages = [sys_msg] + messages

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = out[0][input_len:]
    return processor.decode(generated, skip_special_tokens=True)

In [ ]:
# with torch.inference_mode():
#     out = model.generate(
#         **inputs,
#         max_new_tokens=max_new_tokens,
#         do_sample=False,
#         top_p=None,    # explicitly unset
#         top_k=None,    # explicitly unset
#     )

NameError: name 'inputs' is not defined

In [ ]:
text_messages = [
    {"role": "user", "content": [
        {"type": "text", "text":
            "A hospital's ED saw 1,240 presentations last week. 68% were seen within "
            "4 hours. Among the remaining, 40% breached due to bed block. "
            "How many patients breached due to bed block? Show your reasoning."
        }
    ]},
]

print(generate(text_messages, max_new_tokens=300))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Here is the step-by-step solution:

**1. Calculate the number of patients seen within 4 hours:**

* Total presentations: 1,240
* Percentage seen within 4 hours: 68%
* Number of patients seen within 4 hours = $1,240 \times 0.68$
* Number of patients seen within 4 hours = $843.2$

Since you cannot have a fraction of a patient, we will proceed with the calculated number, or round to the nearest whole number if context demands it. For precise calculation based on the percentages given, we use 843.2.

**2. Calculate the remaining patients (those *not* seen within 4 hours):**

* Percentage remaining = $100\% - 68\% = 32\%$
* Number of remaining patients = $1,240 \times 0.32$
* Number of remaining patients = $396.8$

*(Alternatively, Remaining patients = Total patients - Patients seen within 4 hours: $1,240 - 843.2 = 396.8$)*

**3. Calculate the number of patients who breached due to bed block:**

* The problem states that $40\%$ *of the remaining patients* breached due to bed block.
* Number

In [ ]:
print(generate(text_messages, max_new_tokens=800, thinking=True))

thought
Here's a thinking process to solve the problem:

1.  **Understand the Goal:** The objective is to find the number of patients who breached due to a bed block.

2.  **Identify the Starting Information:**
    *   Total presentations last week: 1,240
    *   Percentage seen within 4 hours: 68%
    *   Percentage of the *remaining* patients who breached due to bed block: 40%

3.  **Step 1: Calculate the number of patients seen within 4 hours.**
    *   Calculate 68% of 1,240.
    *   Calculation: $1240 \times 0.68$
    *   $1240 \times 0.68 = 843.2$
    *   *Self-Correction/Refinement:* Since we are dealing with patients, the result should ideally be an integer. However, in percentage problems, we often carry the decimal until the final step, or assume the input numbers are exact representations. Let's proceed with 843.2 for now, but keep in mind that if the context demands whole numbers, we might need to round (though usually, in these types of math problems, the numbers work out 

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x7bfac6a92700>